In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.models.train import train
metadata = train(n_trials=25, timeout=300)

In [ ]:
import pandas as pd
pd.Series(metadata["metrics"]).to_frame("value")

In [ ]:
pd.Series(metadata["params"]).to_frame("value")

In [ ]:
pd.Series(metadata["feature_importance_gain"]).sort_values(ascending=False).to_frame("gain")

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import precision_recall_curve
from src.config import MODEL_FEATURES, MODEL_PATH, TARGET
from src.models.train import load_dataset, time_split
_, _, test_df = time_split(load_dataset())
booster = lgb.Booster(model_file=str(MODEL_PATH))
scores = booster.predict(test_df[MODEL_FEATURES])
precision, recall, thresholds = precision_recall_curve(test_df[TARGET], scores)
pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]}).iloc[::max(1, len(thresholds) // 15)].round(4)

In [ ]:
test_df.assign(score=scores).groupby(pd.qcut(scores, 10, labels=False, duplicates="drop"))[TARGET].agg(["count", "sum", "mean"]).rename(columns={"sum": "fraud", "mean": "fraud_rate"})